
# Cluster Prioritisation Dashboard

Rank and filter clusters based on inflow magnitude, sell activity, and recurrence across unlocks.



In [36]:

from __future__ import annotations

import csv
import json
from pathlib import Path
from typing import Dict, List

import pandas as pd

def resolve_data_dir() -> Path:
    search = [Path.cwd()] + list(Path.cwd().parents)
    for base in search:
        if (base / 'sui_insiders' / 'data').exists():
            return (base / 'sui_insiders' / 'data').resolve()
        if base.name == 'sui_insiders' and (base / 'data').exists():
            return (base / 'data').resolve()
        if (base / 'data').exists() and (base / 'scripts').exists():
            return (base / 'data').resolve()
    return (Path.cwd() / 'data').resolve()

DATA_DIR = resolve_data_dir()
print(f'Data directory: {DATA_DIR}')



Data directory: /Users/kevincai/Library/Mobile Documents/com~apple~CloudDocs/Career/Interviews/scimitar_capital/sui_insiders/data


In [37]:

cluster_json = DATA_DIR / 'cluster_results.json'
cluster_sales_summary = DATA_DIR / 'cluster_sales_summary.csv'
repeated_addresses_cache = DATA_DIR / 'repeated_addresses.csv'

if not cluster_json.exists():
    raise FileNotFoundError(f'Missing cluster JSON: {cluster_json}')
if not cluster_sales_summary.exists():
    raise FileNotFoundError(f'Missing sales summary CSV: {cluster_sales_summary}')

CLUSTER_RESULTS: Dict[str, Dict[str, object]] = json.loads(cluster_json.read_text())
SALES_SUMMARY_DF = pd.read_csv(cluster_sales_summary) if cluster_sales_summary.exists() else pd.DataFrame()
print(f'Loaded {len(CLUSTER_RESULTS)} clusters; sales summary rows: {len(SALES_SUMMARY_DF)}')



Loaded 165 clusters; sales summary rows: 100


In [38]:

# Identify repeated recipients across unlock CSVs.
unlock_files = sorted(DATA_DIR.glob('*_unlock.csv'))
address_occurrences: Dict[str, set] = {}
for csv_path in unlock_files:
    with csv_path.open('r', encoding='utf-8') as fh:
        reader = csv.DictReader(fh)
        for row in reader:
            addr = row.get('recipient_address')
            if not addr:
                continue
            address_occurrences.setdefault(addr, set()).add(csv_path.name)

repeated_addresses = [
    {'recipient_address': addr, 'occurrences': len(files)}
    for addr, files in address_occurrences.items()
    if len(files) > 1
]
REPEATED_DF = pd.DataFrame(repeated_addresses)
print(f'Repeated recipients: {len(REPEATED_DF)}')



Repeated recipients: 108


In [39]:

cluster_rows = []
for seed, summary in CLUSTER_RESULTS.items():
    cluster_rows.append({
        'seed_wallet': seed,
        'cluster_size': summary.get('cluster_size', len(summary.get('wallets', []) or [])),
        'total_cluster_inflow_sui': summary.get('total_cluster_inflow_sui', 0.0),
        'wallets': summary.get('wallets', []),
    })

CLUSTERS_DF = pd.DataFrame(cluster_rows)
print(f'Cluster summary rows: {len(CLUSTERS_DF)}')
CLUSTERS_DF.head()



Cluster summary rows: 165


,seed_wallet,cluster_size,total_cluster_inflow_sui,wallets
0,0x3444b3ee072e398e5db552ef69bdaf5ebb4b7500b908...,63,1.495680e+06,[0x06fa64721e053471c1ed1e31bd5cc941e5819b737da...
1,0x0d7f0b6f3cc9e0841eedbf0ec7660abce18899a95141...,63,1.495680e+06,[0x06fa64721e053471c1ed1e31bd5cc941e5819b737da...
2,0x5920ab542d27a01a49ec26c90421717a2a68c33ce004...,111,1.543158e+06,[0x067b85a24a48a04408e8ff377d8b315a2bbd9009cd9...
3,0x676087d52dfdcb981202d6f129f0259bdc5f3a1c90a4...,111,1.543158e+06,[0x067b85a24a48a04408e8ff377d8b315a2bbd9009cd9...
4,0x935029ca5219502a47ac9b69f556ccf6e2198b5e7815...,147,3.710841e+06,[0x009c3127c021f2b4d675f0cd4e6020713fd79afd006...


In [40]:

merged_df = CLUSTERS_DF.merge(
    SALES_SUMMARY_DF,
    how='left',
    on='seed_wallet',
    suffixes=('', '_sales')
)

if not REPEATED_DF.empty:
    merged_df['repeated_recipient_flag'] = merged_df['seed_wallet'].isin(REPEATED_DF['recipient_address'])
else:
    merged_df['repeated_recipient_flag'] = False

merged_df[['total_sui_sold', 'sale_count']] = merged_df[['total_sui_sold', 'sale_count']].fillna(0)
merged_df['cluster_size'] = merged_df['cluster_size'].fillna(1)
print(f'Merged DataFrame rows: {len(merged_df)}')
merged_df.head()



Merged DataFrame rows: 165


,seed_wallet,cluster_size,total_cluster_inflow_sui,wallets,first_sale,last_sale,total_sui_sold,sale_count,repeated_recipient_flag
0,0x3444b3ee072e398e5db552ef69bdaf5ebb4b7500b908...,63,1.495680e+06,[0x06fa64721e053471c1ed1e31bd5cc941e5819b737da...,2024-05-02 05:22:47.254000+00:00,2024-08-01 13:03:47.815000+00:00,78998.011037,20.0,False
1,0x0d7f0b6f3cc9e0841eedbf0ec7660abce18899a95141...,63,1.495680e+06,[0x06fa64721e053471c1ed1e31bd5cc941e5819b737da...,2024-05-02 05:22:47.254000+00:00,2024-08-01 13:03:47.815000+00:00,78998.011037,20.0,False
2,0x5920ab542d27a01a49ec26c90421717a2a68c33ce004...,111,1.543158e+06,[0x067b85a24a48a04408e8ff377d8b315a2bbd9009cd9...,2024-05-02 05:20:42.955000+00:00,2024-08-31 23:33:01.274000+00:00,118071.627274,11.0,False
3,0x676087d52dfdcb981202d6f129f0259bdc5f3a1c90a4...,111,1.543158e+06,[0x067b85a24a48a04408e8ff377d8b315a2bbd9009cd9...,2024-05-02 05:20:42.955000+00:00,2024-08-31 23:33:01.274000+00:00,118071.627274,11.0,False
4,0x935029ca5219502a47ac9b69f556ccf6e2198b5e7815...,147,3.710841e+06,[0x009c3127c021f2b4d675f0cd4e6020713fd79afd006...,2024-05-02 05:20:42.955000+00:00,2024-08-31 23:33:01.274000+00:00,185945.945177,14.0,True


In [41]:

MIN_CLUSTER_INFLOWS = 5000
PRIORITY_PORTION = 25

ranked_df = merged_df.sort_values(
    ['total_cluster_inflow_sui', 'total_sui_sold', 'sale_count'],
    ascending=False
)
ranked_df = ranked_df.reset_index(drop=True)
ranked_df['priority_score'] = (
    ranked_df['total_cluster_inflow_sui'] + ranked_df['total_sui_sold'] * 2
)
ranked_df.head()



,seed_wallet,cluster_size,total_cluster_inflow_sui,wallets,first_sale,last_sale,total_sui_sold,sale_count,repeated_recipient_flag,priority_score
0,0x63fbe037f5cca825dbb66db0464b34ece8ad64413cc0...,3,1.615770e+07,[0x63fbe037f5cca825dbb66db0464b34ece8ad64413cc...,2024-06-01 15:50:17.412000+00:00,2024-08-31 23:35:08.126000+00:00,3.077466e+06,32.0,True,2.231263e+07
1,0xecad8b6df613da99ebb12a2d42dffc61503628f0170d...,3,1.615770e+07,[0x63fbe037f5cca825dbb66db0464b34ece8ad64413cc...,2024-06-01 15:50:17.412000+00:00,2024-08-31 23:35:08.126000+00:00,3.077466e+06,32.0,True,2.231263e+07
2,0xcf30da961e1d310546f4df6dcbac0ace091852c77ed2...,3,1.615770e+07,[0x63fbe037f5cca825dbb66db0464b34ece8ad64413cc...,2024-06-01 15:50:17.412000+00:00,2024-08-31 23:35:08.126000+00:00,3.077466e+06,32.0,True,2.231263e+07
3,0xab73ad38c63f83eda02182422b545395be1d3caeb54b...,81,4.431998e+06,[0x06fa64721e053471c1ed1e31bd5cc941e5819b737da...,2024-05-02 05:22:47.254000+00:00,2024-08-01 13:03:47.815000+00:00,1.041751e+05,24.0,True,4.640348e+06
4,0x8b302e06dd0bc3e09f1e009e1dc32a04f6b4b75ab550...,31,4.360950e+06,[0x0352c3aba045fb9cf2b65aa131b8034eddfb6aae9f0...,2024-05-02 05:29:30.617000+00:00,2024-08-31 23:39:47.156000+00:00,2.921775e+04,7.0,True,4.419386e+06


In [42]:

top_clusters = ranked_df[ranked_df['total_cluster_inflow_sui'] >= MIN_CLUSTER_INFLOWS].copy()
if PRIORITY_PORTION:
    limit = max(1, len(top_clusters) * PRIORITY_PORTION // 100)
    top_clusters = top_clusters.head(limit)

print(f'Top clusters to investigate: {len(top_clusters)}')
if len(top_clusters) > 0:
    display(top_clusters[['seed_wallet', 'cluster_size', 'total_cluster_inflow_sui', 'total_sui_sold', 'sale_count', 'repeated_recipient_flag', 'priority_score']])
else:
    print('No clusters met the inflow threshold.')



Top clusters to investigate: 37


,seed_wallet,cluster_size,total_cluster_inflow_sui,total_sui_sold,sale_count,repeated_recipient_flag,priority_score
0,0x63fbe037f5cca825dbb66db0464b34ece8ad64413cc0...,3,1.615770e+07,3.077466e+06,32.0,True,2.231263e+07
1,0xecad8b6df613da99ebb12a2d42dffc61503628f0170d...,3,1.615770e+07,3.077466e+06,32.0,True,2.231263e+07
2,0xcf30da961e1d310546f4df6dcbac0ace091852c77ed2...,3,1.615770e+07,3.077466e+06,32.0,True,2.231263e+07
3,0xab73ad38c63f83eda02182422b545395be1d3caeb54b...,81,4.431998e+06,1.041751e+05,24.0,True,4.640348e+06
4,0x8b302e06dd0bc3e09f1e009e1dc32a04f6b4b75ab550...,31,4.360950e+06,2.921775e+04,7.0,True,4.419386e+06
5,0x1c9deef7e10b88cca4211e5d688bb496d6843d916948...,1,3.962700e+06,1.683524e+06,7.0,True,7.329748e+06
6,0xcd905a5e5e21c07fcdf275588012e5248bcf3a535eff...,137,3.757480e+06,1.413071e+05,35.0,True,4.040095e+06
7,0x2e4706e435195185d69597fdb69bcab9e57a0fa297a1...,15,3.718490e+06,3.373531e+04,8.0,True,3.785960e+06
8,0x344fb28d6b69274ec2ca62ccd40ef03eb01cec68495d...,1,3.718354e+06,0.000000e+00,0.0,False,3.718354e+06
9,0x935029ca5219502a47ac9b69f556ccf6e2198b5e7815...,147,3.710841e+06,1.859459e+05,14.0,True,4.082733e+06


In [43]:
from collections import Counter

cluster_counts = Counter()
wallet_inflows = Counter()
for row in cluster_rows:
    wallets = row.get('wallets') or []
    total_inflow = row.get('total_cluster_inflow_sui', 0.0)
    for wallet in wallets:
        cluster_counts[wallet] += 1
        wallet_inflows[wallet] += total_inflow

common_wallets = pd.DataFrame([
    {
        'wallet': wallet,
        'cluster_count': count,
        'aggregate_cluster_inflow_sui': wallet_inflows[wallet],
    }
    for wallet, count in cluster_counts.items()
    if count > 1
]).sort_values(['cluster_count', 'aggregate_cluster_inflow_sui'], ascending=False)

print(f'Wallets appearing in multiple clusters: {len(common_wallets)}')
display(common_wallets.head(20))

if not common_wallets.empty:
    shared_path = DATA_DIR / 'cluster_shared_wallets.csv'
    common_wallets.to_csv(shared_path, index=False)
    print(f'Saved shared-wallet list to {shared_path}')


Wallets appearing in multiple clusters: 440


,wallet,cluster_count,aggregate_cluster_inflow_sui
46,0xcd905a5e5e21c07fcdf275588012e5248bcf3a535eff...,28,5.517284e+07
367,0x341fa71e4e58d63668034125c3152f935b00b0bb5c68...,25,5.726516e+06
369,0xc47fbfc42ecd279d4d76303b0549c8312ab32bbbcbd6...,25,5.726516e+06
371,0x0c174066a83ec59add25e17b7cabc439f75418904ac3...,23,5.041228e+06
372,0x24bcc073dca6666c1b8240585da641cf400884fa4b08...,23,5.041228e+06
373,0x315f07413f47aae951765e7898d0a5a7f08467ac8621...,23,5.041228e+06
374,0x31ed7ca5dcbccd6bd2837c5624abd2915458998d8830...,23,5.041228e+06
375,0x3309bff9e6c5ce7439b19694b7374a223db854cc5f1c...,23,5.041228e+06
376,0x37b88028a81b091c0eacd09e8bc711f6fa65b16dd4ba...,23,5.041228e+06
377,0x4961c695fb5e9f1d4f7e1e7c347588973ddcd8818665...,23,5.041228e+06


Saved shared-wallet list to /Users/kevincai/Library/Mobile Documents/com~apple~CloudDocs/Career/Interviews/scimitar_capital/sui_insiders/data/cluster_shared_wallets.csv


In [44]:

if not REPEATED_DF.empty:
    overlapped = REPEATED_DF[REPEATED_DF['recipient_address'].isin(merged_df['seed_wallet'])]
    overlapped = overlapped.sort_values('occurrences', ascending=False)
    print(f'Repeated recipients that are seed wallets: {len(overlapped)}')
    display(overlapped)
else:
    print('No repeated recipients detected.')



Repeated recipients that are seed wallets: 25


,recipient_address,occurrences
16,0xcf30da961e1d310546f4df6dcbac0ace091852c77ed2...,5
30,0xf8b7b95d01ae79756fc3d1bc58675e59a17f07cdafe3...,5
6,0xecad8b6df613da99ebb12a2d42dffc61503628f0170d...,5
4,0x63fbe037f5cca825dbb66db0464b34ece8ad64413cc0...,4
43,0x13c075644dad21bf10cb12447b825b535c24ddae428e...,4
79,0x2e4706e435195185d69597fdb69bcab9e57a0fa297a1...,4
73,0xc8f62180c9e10dacb5d72f1bb407c6e6f09bfa9b75ca...,4
45,0xab73ad38c63f83eda02182422b545395be1d3caeb54b...,4
15,0x60dd01bc037e2c1ea2aaf02187701f9f4453ba323338...,4
35,0x935029ca5219502a47ac9b69f556ccf6e2198b5e7815...,3



## Next Steps

- Adjust thresholds (`MIN_CLUSTER_INFLOWS`, `PRIORITY_PORTION`) and rerun to dial the workload.
- Use `cluster_graphs.ipynb` to visualise any seed wallet surfaced here.
- Cross-reference with `cluster_sales_detail.csv` to inspect individual outflows.

